In [2]:
from pyspark.sql import SparkSession

# Creating Spark Session
spark = (SparkSession.builder
    .appName("BigData_Lab02.2_Ubuntu") # Spark session name
    .master("local[*]") \
    .config("spark.driver.bindAddress", "127.0.0.1")\
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.ui.port", "4041")
    .config("spark.port.maxRetries", "100")
    .config("spark.driver.memory", "2g")
    .config("spark.jars.packages", 
            "org.apache.spark:spark-sql-kafka-0-10_2.13:4.0.1,"
            "org.apache.kafka:kafka-clients:3.6.0,"
            "org.apache.spark:spark-streaming-kafka-0-10_2.13:4.0.1") 
    .getOrCreate())

# Checking success
print(f"Successfully! Spark version: {spark.version}")

Successfully! Spark version: 4.0.1


In [3]:
import kagglehub

# Download dataset from kaggle
path = kagglehub.dataset_download("grouplens/movielens-latest-small")

# reading data to spark frame
df_ratings = spark.read.csv(path + "/ratings.csv", header=True, inferSchema=True)
df_movies = spark.read.csv(path + "/movies.csv", header=True, inferSchema=True)
df_tags = spark.read.csv(path + "/tags.csv", header=True, inferSchema=True)

/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from pyspark.sql.types import *

rating_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("rating", FloatType()),
    StructField("timestamp", LongType())
])

movie_schema = StructType([
    StructField("movieId", IntegerType()),
    StructField("title", StringType()),
    StructField("genres", StringType())
])

tag_schema = StructType([
    StructField("userId", IntegerType()),
    StructField("movieId", IntegerType()),
    StructField("tag", StringType()),
    StructField("timestamp", LongType())
])

In [5]:
from pyspark.sql.functions import *
brokers = "localhost:9092,localhost:9192,localhost:9292"
def read_kafka_topic(topic_name, schema):
    return spark.readStream \
        .format("kafka") \
        .option("kafka.bootstrap.servers", brokers) \
        .option("subscribe", topic_name) \
        .option("startingOffsets", "earliest") \
        .option("maxOffsetsPerTrigger", 100) \
        .load() \
        .selectExpr("CAST(value AS STRING)") \
        .select(from_json(col("value"), schema).alias("data")) \
        .select("data.*") \
        .withColumn("timestamp", to_timestamp(from_unixtime(col("timestamp"))))

# Đọc luồng dữ liệu
ratings_stream = read_kafka_topic("Lab1_ratings", rating_schema)
tags_stream = read_kafka_topic("Lab1_tags", tag_schema)

# Movies thường là dữ liệu tĩnh để join trong Exercise 2 & 3 [cite: 208]
movies_static = df_movies

In [ ]:
hot_genres = ratings_stream.join(movies_static, "movieId") \
    .withColumn("genre", explode(split(col("genres"), "\|"))) \
    .groupBy("genre") \
    .count() \
    .orderBy(col("count").desc())

# 2. Khởi chạy luồng stream [cite: 185, 186]
query2 = hot_genres.writeStream \
    .outputMode("complete") \
    .format("console") \
    .option("truncate", "false") \
    .trigger(processingTime='5 seconds') \
    .start() # [cite: 187, 188, 189]

print(">>> Start Exercise 2 in 60 second...")
query2.awaitTermination(60) 

# 4. Dừng luồng sau khi hết thời gian timeout
query2.stop()
print(">>> Stop query 2.")

In [11]:
# Lấy danh sách tất cả các query đang hoạt động trong SparkSession
active_queries = spark.streams.active

if len(active_queries) > 0:
    print(f"Đang tìm thấy {len(active_queries)} query đang chạy. Tiến hành dừng...")
    for q in active_queries:
        print(f"Đã dừng query: {q.name if q.name else q.id}")
        q.stop()
else:
    print("Không có query nào đang chạy ngầm.")

Không có query nào đang chạy ngầm.


## EX3

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# 1. Tính toán số lượt rate theo cửa sổ 5 phút (Tumbling Window)
# Phải đặt watermark TRƯỚC khi thực hiện groupBy window 
windowed_counts = ratings_stream \
    .withWatermark("timestamp", "10 minutes") \
    .groupBy(
        window(col("timestamp"), "5 minutes"), # Cửa sổ 5 phút 
        col("movieId")
    ) \
    .count() \
    .withColumnRenamed("count", "cnt")

# 2. Join với bảng movies tĩnh để lấy tiêu đề phim [cite: 208, 209]
trending_df = windowed_counts.join(movies_static, "movieId")

# 3. Hàm xử lý xếp hạng cho từng Micro-Batch
def process_trending_batch(batch_df, batch_id):
    # Định nghĩa quy tắc xếp hạng:
    # Phân vùng theo window, sắp xếp theo lượt rate giảm dần,movieId tăng dần [cite: 209, 213]
    window_spec = Window.partitionBy("window").orderBy(col("cnt").desc(), col("movieId").asc())
    
    # Thực hiện xếp hạng và lọc Top 3 
    final_result = batch_df.withColumn("rank", dense_rank().over(window_spec)) \
        .filter(col("rank") <= 3) \
        .select("window", "movieId", "title", "cnt", "rank") \
        .orderBy("window", "rank")
    
    if not final_result.isEmpty():
        print(f"\n--- Batch ID: {batch_id} | Trending Now (Top 3) ---")
        final_result.show(truncate=False)

# 4. Khởi chạy stream với trigger 5 giây [cite: 204]
query3 = trending_df.writeStream \
    .foreachBatch(process_trending_batch) \
    .trigger(processingTime='5 seconds') \
    .start()

# Chạy demo trong 20 giây rồi dừng
print(">>> Đang chạy Exercise 3 trong 60 giây...")
query3.awaitTermination()
# query3.stop()
# print(">>> Demo Exercise 3 kết thúc.")

26/04/27 21:27:52 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-e9280aa1-fb9f-4097-83e7-40f8e2234e97. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 21:27:52 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


>>> Đang chạy Exercise 3 trong 60 giây...


26/04/27 21:28:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 10281 milliseconds



--- Batch ID: 1 | Trending Now (Top 3) ---


26/04/27 21:28:13 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 11459 milliseconds


...

26/04/27 21:28:36 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 11420 milliseconds


+------------------------------------------+-------+------------------------------------------------------+---+----+
|window                                    |movieId|title                                                 |cnt|rank|
+------------------------------------------+-------+------------------------------------------------------+---+----+
|{2000-08-08 14:25:00, 2000-08-08 14:30:00}|2492   |20 Dates (1998)                                       |1  |1   |
|{2001-04-10 03:35:00, 2001-04-10 03:40:00}|106    |Nobody Loves Me (Keiner liebt mich) (1994)            |1  |1   |
|{2001-04-10 03:35:00, 2001-04-10 03:40:00}|595    |Beauty and the Beast (1991)                           |1  |2   |
|{2001-04-10 03:40:00, 2001-04-10 03:45:00}|126    |NeverEnding Story III, The (1994)                     |1  |1   |
|{2001-04-10 03:40:00, 2001-04-10 03:45:00}|247    |Heavenly Creatures (1994)                             |1  |2   |
|{2001-04-10 03:40:00, 2001-04-10 03:45:00}|450    |With Honors 

26/04/27 21:28:47 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5239 milliseconds
ERROR:root:KeyboardInterrupt while sending command.             (85 + 12) / 200]
Traceback (most recent call last):
  File "/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/kiennguyen/bigdata/lab01/venv/lib/python3.12/site-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/socket.py", line 707, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

26/04/27 21:28:52 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 5727 milliseconds


## EX4

In [ ]:
from pyspark.sql.functions import *

# 1. Thiết lập Watermark cho cả hai luồng (Yêu cầu bắt buộc) 
ratings_with_wm = ratings_stream.withWatermark("timestamp", "10 minutes")
tags_with_wm = tags_stream.withWatermark("timestamp", "10 minutes")

# 2. Thực hiện Stream-Stream Join trên movieId [cite: 220, 225]
# Điều kiện: movieId trùng nhau VÀ thời gian tag không cách quá 5 phút so với rating
joined_stream = ratings_with_wm.alias("r").join(
    tags_with_wm.alias("t"),
    expr("""
        r.movieId = t.movieId AND
        t.timestamp >= r.timestamp - interval 5 minutes AND
        t.timestamp <= r.timestamp + interval 5 minutes
    """),
    "inner"
)

# 3. Chọn các cột đầu ra (movieId, tag, rating, ratingTime, tagTime) [cite: 221]
result_ex4 = joined_stream.select(
    col("r.movieId"),
    col("t.tag"),
    col("r.rating"),
    col("r.timestamp").alias("ratingTime"),
    col("t.timestamp").alias("tagTime")
)

# 4. Ghi kết quả ra console ở chế độ APPEND mỗi 5 giây [cite: 216, 217, 221]
query4 = result_ex4.writeStream \
    .outputMode("append") \
    .format("console") \
    .trigger(processingTime='5 seconds') \
    .start()

print(">>> Đang chạy Exercise 4 (Stream-Stream Join) trong 60 giây...")
query4.awaitTermination()
# query4.stop()
# print(">>> Hoàn thành toàn bộ Lab 02.")

26/04/27 21:39:04 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-92254ad0-1c90-4cd4-9393-8b2cd9a193d3. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/27 21:39:04 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


>>> Đang chạy Exercise 4 (Stream-Stream Join) trong 60 giây...


26/04/27 21:39:21 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 16704 milliseconds


-------------------------------------------
Batch: 0
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



-------------------------------------------
Batch: 1
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



...

-------------------------------------------
Batch: 17
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



26/04/27 21:43:51 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 11249 milliseconds


-------------------------------------------
Batch: 18
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+



26/04/27 21:44:02 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000} milliseconds, but spent 11168 milliseconds


-------------------------------------------
Batch: 19
-------------------------------------------
+-------+---+------+----------+-------+
|movieId|tag|rating|ratingTime|tagTime|
+-------+---+------+----------+-------+
+-------+---+------+----------+-------+

